# CP_21 — archived capstone notebook

Run from the repository root. Stored outputs are historical evidence, not a fresh execution. CP_20 and CP_21 setup paths were made portable; select your own project kernel. Earlier notebooks retain their original code. See docs/REPRODUCIBILITY.md.

In [1]:
# CP_21 — Round 10
# Check the Python environment and locate the project data.

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

project_path = Path.cwd()

expected_python = Path(sys.executable)

initial_data_path = project_path / "initial_data"
feedback_path = project_path / "week_9"

inputs_path = feedback_path / "inputs.txt"
outputs_path = feedback_path / "outputs.txt"

completed_rounds = 9
submission_round = 10

correct_kernel = (
    Path(sys.executable).resolve() == expected_python.resolve()
)

print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())
print("Project folder:", project_path)

print("\nExpected virtual environment selected:", correct_kernel)
print("Project folder found:", project_path.is_dir())
print("Initial data folder found:", initial_data_path.is_dir())
print("Feedback folder found:", feedback_path.is_dir())
print("Cumulative inputs found:", inputs_path.is_file())
print("Cumulative outputs found:", outputs_path.is_file())

print("\nExpected completed rounds:", completed_rounds)
print("Round to prepare:", submission_round)

assert correct_kernel, (
    "Select the notebook kernel corresponding to: "
    f"{expected_python}"
)

assert project_path.is_dir(), (
    f"Project folder not found: {project_path}"
)

assert initial_data_path.is_dir(), (
    f"Initial data folder not found: {initial_data_path}"
)

assert feedback_path.is_dir(), (
    f"Feedback folder not found: {feedback_path}"
)

assert inputs_path.is_file(), (
    f"Inputs file not found: {inputs_path}"
)

assert outputs_path.is_file(), (
    f"Outputs file not found: {outputs_path}"
)

print("\nEnvironment and file checks passed.")
print(
    f"Ready to load and validate the "
    f"{completed_rounds} feedback rounds."
)

Python executable: c:\Users\AG\BeatrizGIT\Master\.venv\Scripts\python.exe
Working directory: c:\Users\AG\BeatrizGIT\Master\CapstoneProject
Project folder: C:\Users\AG\BeatrizGIT\Master\CapstoneProject

Expected virtual environment selected: True
Project folder found: True
Initial data folder found: True
Feedback folder found: True
Cumulative inputs found: True
Cumulative outputs found: True

Expected completed rounds: 9
Round to prepare: 10

Environment and file checks passed.
Ready to load and validate the 9 feedback rounds.


### Environment and data locations

This cell checks that the notebook uses the project's virtual environment
and that the initial data and cumulative feedback files are available.

The notebook prepares submission round 10 using feedback from nine completed
rounds. These round numbers are declared here; the file contents will be
checked in the next cell.

Explicit paths and environment checks help make the workflow reproducible.
Another researcher would need the same data files and compatible package
versions, with the paths adjusted to their own environment.

In [2]:
# CP_21 — Round 10
# Load and validate the initial data and nine completed feedback rounds.

import ast
import re

import numpy as np


def read_feedback_rounds(path):
    """Read cumulative feedback safely without executing file contents."""
    text = path.read_text(encoding="utf-8-sig").strip()

    # Convert NumPy-style array([...]) representations into plain lists.
    text = re.sub(r"array\((\[[^\[\]]*\])\)", r"\1", text)

    tree = ast.parse(text, mode="exec")
    rounds = []

    for node in tree.body:
        if not isinstance(node, ast.Expr):
            raise ValueError(
                f"Unexpected content in {path.name}: "
                "expected one list per feedback round."
            )

        rounds.append(ast.literal_eval(node.value))

    return rounds


# Check that the first code cell has been executed.
required_variables = [
    "initial_data_path",
    "inputs_path",
    "outputs_path",
    "completed_rounds",
    "submission_round",
]

missing_variables = [
    name for name in required_variables if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the first code cell before this cell. "
        f"Missing variables: {', '.join(missing_variables)}"
    )


expected_dimensions = {
    1: 2,
    2: 2,
    3: 3,
    4: 4,
    5: 4,
    6: 5,
    7: 6,
    8: 8,
}

expected_initial_counts = {
    1: 10,
    2: 10,
    3: 15,
    4: 30,
    5: 20,
    6: 20,
    7: 30,
    8: 40,
}


# Load the cumulative feedback files.
input_rounds = read_feedback_rounds(inputs_path)
output_rounds = read_feedback_rounds(outputs_path)

assert len(input_rounds) == completed_rounds, (
    f"Expected {completed_rounds} input rounds, "
    f"but found {len(input_rounds)}."
)

assert len(output_rounds) == completed_rounds, (
    f"Expected {completed_rounds} output rounds, "
    f"but found {len(output_rounds)}."
)


# Validate every feedback input–output pair.
for round_number, (round_inputs, round_outputs) in enumerate(
    zip(input_rounds, output_rounds),
    start=1,
):
    assert len(round_inputs) == 8, (
        f"Round {round_number}: expected inputs for eight functions."
    )

    assert len(round_outputs) == 8, (
        f"Round {round_number}: expected outputs for eight functions."
    )

    for function, dimension in expected_dimensions.items():
        x = np.asarray(round_inputs[function - 1], dtype=float)
        y = np.asarray(round_outputs[function - 1], dtype=float)

        assert x.shape == (dimension,), (
            f"Round {round_number}, Function {function}: "
            f"expected {dimension} coordinates, found shape {x.shape}."
        )

        assert y.ndim == 0, (
            f"Round {round_number}, Function {function}: "
            "expected one scalar output."
        )

        assert np.all(np.isfinite(x)) and np.isfinite(y).item(), (
            f"Round {round_number}, Function {function}: "
            "non-finite values detected."
        )

        assert np.all((x >= 0) & (x < 1)), (
            f"Round {round_number}, Function {function}: "
            "feedback coordinates must satisfy 0 <= x < 1."
        )


# Rebuild the datasets from the original files on every execution.
# This prevents appending the same feedback twice when rerunning the cell.
loaded_initial_data = {}
loaded_updated_data = {}

for function, dimension in expected_dimensions.items():
    function_path = initial_data_path / f"function_{function}"

    initial_inputs_file = function_path / "initial_inputs.npy"
    initial_outputs_file = function_path / "initial_outputs.npy"

    assert initial_inputs_file.is_file(), (
        f"File not found: {initial_inputs_file}"
    )

    assert initial_outputs_file.is_file(), (
        f"File not found: {initial_outputs_file}"
    )

    X_initial = np.asarray(
        np.load(initial_inputs_file, allow_pickle=False),
        dtype=float,
    )

    y_initial = np.asarray(
        np.load(initial_outputs_file, allow_pickle=False),
        dtype=float,
    ).reshape(-1)

    expected_count = expected_initial_counts[function]

    assert X_initial.shape == (expected_count, dimension), (
        f"Function {function}: unexpected initial input shape "
        f"{X_initial.shape}."
    )

    assert y_initial.shape == (expected_count,), (
        f"Function {function}: unexpected initial output shape "
        f"{y_initial.shape}."
    )

    assert np.all(np.isfinite(X_initial)), (
        f"Function {function}: non-finite initial inputs."
    )

    assert np.all(np.isfinite(y_initial)), (
        f"Function {function}: non-finite initial outputs."
    )

    X_feedback = np.asarray(
        [values[function - 1] for values in input_rounds],
        dtype=float,
    )

    y_feedback = np.asarray(
        [values[function - 1] for values in output_rounds],
        dtype=float,
    )

    loaded_initial_data[function] = {
        "X": X_initial.copy(),
        "y": y_initial.copy(),
    }

    loaded_updated_data[function] = {
        "X": np.vstack([X_initial, X_feedback]),
        "y": np.concatenate([y_initial, y_feedback]),
    }


# Make the datasets available only after all eight pass validation.
initial_data = loaded_initial_data
updated_data = loaded_updated_data

for function, dimension in expected_dimensions.items():
    initial_count = len(initial_data[function]["y"])
    updated_count = len(updated_data[function]["y"])

    print(
        f"Function {function}: "
        f"{initial_count} → {updated_count} observations "
        f"| {dimension} variables"
    )

print(f"\nValidated feedback rounds: {len(input_rounds)}")
print(f"Round to prepare: {submission_round}")
print("All eight datasets loaded and validated successfully.")

Function 1: 10 → 19 observations | 2 variables
Function 2: 10 → 19 observations | 2 variables
Function 3: 15 → 24 observations | 3 variables
Function 4: 30 → 39 observations | 4 variables
Function 5: 20 → 29 observations | 4 variables
Function 6: 20 → 29 observations | 5 variables
Function 7: 30 → 39 observations | 6 variables
Function 8: 40 → 49 observations | 8 variables

Validated feedback rounds: 9
Round to prepare: 10
All eight datasets loaded and validated successfully.


### Data loading and validation

This cell combines the original observations with nine completed feedback
rounds. After successful validation, the datasets contain 19, 19, 24, 39,
29, 29, 39 and 49 observations for Functions 1–8, respectively.

The checks verify the number of feedback rounds, the number of functions
per round, input dimensionality, finite numerical values and permitted
feedback coordinate ranges. Each output must be a single scalar value.
Negative outputs are retained because all eight functions are maximisation
problems.

The datasets are rebuilt from the original files on every execution,
preventing repeated execution from appending the same feedback again.
Initial observations remain separately available for tracking improvements
against the original baseline.

For transparency, the initial data and sequential portal feedback are
treated as distinct data sources. Later queries were selected adaptively
using previous results, so the combined datasets should not be interpreted
as uniform or independent random samples of the search space.

In [3]:
# CP_21 — Round 10
# Review round 9 results and track improvements across completed rounds.

import numpy as np
import pandas as pd
from IPython.display import display

latest_round = completed_rounds
feedback_outputs = np.asarray(output_rounds, dtype=float)

assert feedback_outputs.shape == (completed_rounds, 8), (
    "Expected one output per function for every completed round."
)

latest_rows = []
history_rows = []

for function in range(1, 9):
    initial_best = float(np.max(initial_data[function]["y"]))
    best_so_far = initial_best

    for round_index in range(completed_rounds):
        round_number = round_index + 1
        observed_output = float(
            feedback_outputs[round_index, function - 1]
        )

        best_before = best_so_far

        # A larger output is better, including when outputs are negative.
        improved = observed_output > best_before
        improvement = max(0.0, observed_output - best_before)
        best_so_far = max(best_before, observed_output)

        history_rows.append({
            "Round": round_number,
            "Function": function,
            "Observed output": observed_output,
            "Best before round": best_before,
            "Improved": improved,
            "Improvement": improvement,
            "Best after round": best_so_far,
        })

        if round_number == latest_round:
            latest_rows.append({
                "Function": function,
                f"Best before round {latest_round}": best_before,
                f"Round {latest_round} output": observed_output,
                f"Round {latest_round} improved": improved,
                "Improvement": improvement,
                "Current best": best_so_far,
            })

    # Confirm that the history agrees with the combined dataset.
    assert best_so_far == float(
        np.max(updated_data[function]["y"])
    ), (
        f"Function {function}: inconsistent best observed output."
    )

improvement_history = pd.DataFrame(history_rows)

round_results = (
    pd.DataFrame(latest_rows)
    .set_index("Function")
)

numeric_columns = [
    f"Best before round {latest_round}",
    f"Round {latest_round} output",
    "Improvement",
    "Current best",
]

display(
    round_results.style.format({
        column: "{:.6e}" for column in numeric_columns
    })
)

print()

for round_number in range(1, completed_rounds + 1):
    improved_functions = improvement_history.loc[
        (improvement_history["Round"] == round_number)
        & improvement_history["Improved"],
        "Function",
    ].tolist()

    print(
        f"Functions improved in round {round_number}: "
        f"{improved_functions} | Count: {len(improved_functions)}"
    )

,Best before round 9,Round 9 output,Round 9 improved,Improvement,Current best
Function,,,,,
1,7.710875e-16,2.204326e-19,False,0.000000e+00,7.710875e-16
2,6.582395e-01,5.369471e-01,False,0.000000e+00,6.582395e-01
3,-3.096391e-02,-8.952091e-02,False,0.000000e+00,-3.096391e-02
4,6.152729e-01,-1.427854e-01,False,0.000000e+00,6.152729e-01
5,8.510289e+03,7.557854e+03,False,0.000000e+00,8.510289e+03
6,-2.424310e-01,-3.038324e-01,False,0.000000e+00,-2.424310e-01
7,2.374721e+00,2.218146e+00,False,0.000000e+00,2.374721e+00
8,9.922955e+00,9.958227e+00,True,3.527205e-02,9.958227e+00



Functions improved in round 1: [2, 4, 5, 7, 8] | Count: 5
Functions improved in round 2: [6, 8] | Count: 2
Functions improved in round 3: [5, 8] | Count: 2
Functions improved in round 4: [6, 8] | Count: 2
Functions improved in round 5: [3, 4, 5, 8] | Count: 4
Functions improved in round 6: [5, 6, 7] | Count: 3
Functions improved in round 7: [5] | Count: 1
Functions improved in round 8: [4, 5, 7] | Count: 3
Functions improved in round 9: [8] | Count: 1


### Round 9 — Observed improvements

Round 9 improved the best observed output for Function 8, increasing it
from approximately 9.922955 to 9.958227. The other seven functions did
not exceed their previous best results.

Function 5 returned approximately 7,557.85, below its previous best of
8,510.29. Function 4 returned approximately -0.142785, compared with a
previous best of 0.615273. These outcomes show that querying near promising
observations does not guarantee further improvement.

The history records every observed output, including unsuccessful queries.
The best-so-far value is retained when a new observation is worse; a zero
improvement therefore means no new record, not that the function output
was zero.

This record supports transparent evaluation by distinguishing actual
improvements from model predictions. The number of functions improved
is useful for tracking progress, but does not measure the magnitude of
the gains or the information obtained from each query. Raw improvements
also should not be compared directly across functions with different
output scales.

The next step compares the predictions recorded in CP_20 with the round 9
observations before fitting models for round 10.

In [4]:
# CP_21 — Round 10
# Compare predictions recorded before round 9 with its observed outputs.
# Historical predictions are rounded values from the displayed CP_20 results.

import numpy as np
import pandas as pd
from IPython.display import display

historical_round = 9

required_variables = [
    "input_rounds",
    "output_rounds",
    "expected_dimensions",
    "completed_rounds",
]

missing_variables = [
    name for name in required_variables if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the data-loading cell first. "
        f"Missing variables: {', '.join(missing_variables)}"
    )

assert completed_rounds >= historical_round, (
    "Round 9 feedback is required for this comparison."
)

# These predictions were made before the round 9 outputs were available.
historical_predictions = {
    1: {
        "query": "0.257861-0.414992",
        "mean": -1.777581e-04,
        "std": 8.715443e-04,
        "auxiliary": np.nan,
    },
    2: {
        "query": "0.693314-0.998760",
        "mean": 5.778382e-01,
        "std": 7.550253e-02,
        "auxiliary": np.nan,
    },
    3: {
        "query": "0.607473-0.016509-0.462216",
        "mean": -3.625537e-02,
        "std": 5.902179e-02,
        "auxiliary": np.nan,
    },
    4: {
        "query": "0.451707-0.435831-0.384426-0.407421",
        "mean": 4.533632e-01,
        "std": 1.861774e-01,
        "auxiliary": np.nan,
    },
    5: {
        "query": "0.999950-0.993086-0.999495-0.943967",
        "mean": 8.391369e03,
        "std": 1.310630e02,
        "auxiliary": 7.743003e03,
    },
    6: {
        "query": "0.458107-0.374813-0.540827-0.692938-0.001140",
        "mean": -3.956728e-01,
        "std": 1.218263e-01,
        "auxiliary": np.nan,
    },
    7: {
        "query": (
            "0.183162-0.134864-0.733336-"
            "0.137261-0.331994-0.699922"
        ),
        "mean": 2.294387,
        "std": 1.189804e-01,
        "auxiliary": 9.289607e-01,
    },
    8: {
        "query": (
            "0.071169-0.178617-0.100685-0.109637-"
            "0.996594-0.554316-0.240640-0.908329"
        ),
        "mean": 9.936421,
        "std": 2.447388e-02,
        "auxiliary": np.nan,
    },
}

comparison_rows = []

for function in range(1, 9):
    record = historical_predictions[function]

    historical_input = np.asarray(
        [float(value) for value in record["query"].split("-")],
        dtype=float,
    )

    feedback_input = np.asarray(
        input_rounds[historical_round - 1][function - 1],
        dtype=float,
    )

    assert historical_input.shape == (
        expected_dimensions[function],
    ), (
        f"Function {function}: incorrect historical query dimensionality."
    )

    assert np.allclose(
        historical_input,
        feedback_input,
        rtol=0,
        atol=1e-12,
    ), (
        f"Function {function}: historical query does not match "
        f"round {historical_round} feedback."
    )

    observed = float(
        output_rounds[historical_round - 1][function - 1]
    )

    predicted_mean = float(record["mean"])
    predicted_std = float(record["std"])
    auxiliary_prediction = float(record["auxiliary"])

    assert np.isfinite(predicted_mean), (
        f"Function {function}: invalid historical predicted mean."
    )

    assert np.isfinite(predicted_std) and predicted_std > 0, (
        f"Function {function}: predicted standard deviation "
        "must be finite and positive."
    )

    residual = observed - predicted_mean
    absolute_error = abs(residual)

    auxiliary_error = (
        abs(observed - auxiliary_prediction)
        if np.isfinite(auxiliary_prediction)
        else np.nan
    )

    comparison_rows.append({
        "Function": function,
        "Predicted mean": predicted_mean,
        "Predicted std": predicted_std,
        "Observed output": observed,
        "Observed minus predicted": residual,
        "Absolute error / predicted std": (
            absolute_error / predicted_std
        ),
        "GP absolute error": absolute_error,
        "Auxiliary prediction": auxiliary_prediction,
        "Auxiliary absolute error": auxiliary_error,
    })

prediction_comparison = (
    pd.DataFrame(comparison_rows)
    .set_index("Function")
)

formats = {
    column: "{:.6e}"
    for column in prediction_comparison.columns
}

formats["Absolute error / predicted std"] = "{:.2f}"

display(
    prediction_comparison.style.format(
        formats,
        na_rep="—",
    )
)

print(
    "\nAll eight historical queries match the round 9 feedback inputs."
)
print("Errors use rounded historical predictions from CP_20.")
print(
    "Error / predicted std is a diagnostic for these individual "
    "observations, not a complete uncertainty calibration assessment."
)

,Predicted mean,Predicted std,Observed output,Observed minus predicted,Absolute error / predicted std,GP absolute error,Auxiliary prediction,Auxiliary absolute error
Function,,,,,,,,
1,-1.777581e-04,8.715443e-04,2.204326e-19,1.777581e-04,0.20,1.777581e-04,—,—
2,5.778382e-01,7.550253e-02,5.369471e-01,-4.089111e-02,0.54,4.089111e-02,—,—
3,-3.625537e-02,5.902179e-02,-8.952091e-02,-5.326554e-02,0.90,5.326554e-02,—,—
4,4.533632e-01,1.861774e-01,-1.427854e-01,-5.961486e-01,3.20,5.961486e-01,—,—
5,8.391369e+03,1.310630e+02,7.557854e+03,-8.335153e+02,6.36,8.335153e+02,7.743003e+03,1.851493e+02
6,-3.956728e-01,1.218263e-01,-3.038324e-01,9.184036e-02,0.75,9.184036e-02,—,—
7,2.294387e+00,1.189804e-01,2.218146e+00,-7.624077e-02,0.64,7.624077e-02,9.289607e-01,1.289186e+00
8,9.936421e+00,2.447388e-02,9.958227e+00,2.180561e-02,0.89,2.180561e-02,—,—



All eight historical queries match the round 9 feedback inputs.
Errors use rounded historical predictions from CP_20.
Error / predicted std is a diagnostic for these individual observations, not a complete uncertainty calibration assessment.


### Round 9 — Prediction errors and uncertainty

This comparison evaluates predictions recorded before round 9 against
the outputs subsequently returned by the portal. The query coordinates
are checked before calculating errors, ensuring that each prediction
is paired with the correct observation. Historical predictions are
rounded values copied from CP_20.

The largest errors relative to predicted uncertainty occur for Functions
5 and 4, at approximately 6.36 and 3.20 predicted standard deviations,
respectively. Both observed outputs are below their GP predictions.
These discrepancies indicate that the predictions were too optimistic
at those specific inputs and that their uncertainty estimates did not
adequately reflect the realised errors.

For Function 5, the auxiliary neural network has a smaller absolute error
than the GP on this observation. For Function 7, the GP is substantially
closer to the observed output than the auxiliary linear regression.
These individual comparisons do not establish that either model is
consistently superior across the search space.

Function 8 exceeds its previous best output, with a prediction error
of approximately 0.89 predicted standard deviations. An accurate
prediction and an improvement over the current best are distinct
evaluation criteria.

For transparency, predictions, observations and errors are retained
separately. These diagnostics concern adaptively selected query points;
they do not constitute a representative test set or a complete
assessment of uncertainty calibration.

In [5]:
# CP_21 — Round 10
# Refit the Gaussian Processes using all observations through round 9.
# Retain the kernel configurations used in CP_20.

import time
import warnings

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel,
)

required_variables = [
    "updated_data",
    "completed_rounds",
    "submission_round",
]

missing_variables = [
    name for name in required_variables if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the preceding setup and data-loading cells first. "
        f"Missing variables: {', '.join(missing_variables)}"
    )

assert completed_rounds == 9 and submission_round == 10, (
    "This cell prepares round 10 using nine completed feedback rounds."
)

# Configurations retained from CP_20.
gp_configurations = {
    1: {
        "name": "Matern_1.5_original",
        "nu": 1.5,
        "upper_bound": 2.0,
    },
    2: {
        "name": "Matern_1.5_wider",
        "nu": 1.5,
        "upper_bound": 10.0,
    },
    3: {
        "name": "Matern_1.5_original",
        "nu": 1.5,
        "upper_bound": 2.0,
    },
    4: {
        "name": "Matern_1.5_wider",
        "nu": 1.5,
        "upper_bound": 10.0,
    },
    5: {
        "name": "Matern_1.5_original",
        "nu": 1.5,
        "upper_bound": 2.0,
    },
    6: {
        "name": "Matern_2.5_original",
        "nu": 2.5,
        "upper_bound": 2.0,
    },
    7: {
        "name": "Matern_2.5_original",
        "nu": 2.5,
        "upper_bound": 2.0,
    },
    8: {
        "name": "Matern_2.5_wider",
        "nu": 2.5,
        "upper_bound": 10.0,
    },
}

fitted_models = {}
fit_rows = []
warning_rows = []

for function in range(1, 9):
    X = np.asarray(updated_data[function]["X"], dtype=float)
    y = np.asarray(updated_data[function]["y"], dtype=float)

    configuration = gp_configurations[function]

    assert X.ndim == 2 and y.shape == (X.shape[0],), (
        f"Function {function}: incompatible input and output shapes."
    )
    assert np.all(np.isfinite(X)) and np.all(np.isfinite(y)), (
        f"Function {function}: non-finite training data."
    )

    kernel = (
        ConstantKernel(
            constant_value=1.0,
            constant_value_bounds=(1e-3, 1e3),
        )
        * Matern(
            length_scale=np.full(X.shape[1], 0.2),
            length_scale_bounds=(
                0.02,
                configuration["upper_bound"],
            ),
            nu=configuration["nu"],
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1),
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=3,
        random_state=42,
    )

    start_time = time.perf_counter()

    # Preserve fitting warnings for inspection below.
    with warnings.catch_warnings(record=True) as captured_warnings:
        warnings.simplefilter("always")
        model.fit(X, y)

    fit_seconds = time.perf_counter() - start_time

    fitted_models[function] = model

    fit_rows.append({
        "Function": function,
        "Observations": len(y),
        "GP configuration": configuration["name"],
        "Fitted kernel": str(model.kernel_),
        "Log-marginal likelihood": (
            model.log_marginal_likelihood_value_
        ),
        "GP fit seconds": fit_seconds,
    })

    for warning in captured_warnings:
        warning_rows.append({
            "Function": function,
            "Category": warning.category.__name__,
            "Message": str(warning.message),
        })

# Make all eight fitted models available to subsequent cells.
gp_models = fitted_models

gp_fit_summary = (
    pd.DataFrame(fit_rows)
    .set_index("Function")
)

gp_fit_warnings = pd.DataFrame(
    warning_rows,
    columns=["Function", "Category", "Message"],
)

display(
    gp_fit_summary.style.format({
        "Log-marginal likelihood": "{:.6f}",
        "GP fit seconds": "{:.3f}",
    })
)

print(
    "\nAll eight Gaussian Processes were fitted "
    "using observations through round 9."
)
print(
    "Kernel configurations were retained from CP_20; "
    "continuous kernel parameters were re-estimated."
)

if gp_fit_warnings.empty:
    print("\nNo fitting warnings were recorded.")
else:
    print(f"\nRecorded fitting warnings: {len(gp_fit_warnings)}")

    with pd.option_context("display.max_colwidth", None):
        display(gp_fit_warnings)

,Observations,GP configuration,Fitted kernel,Log-marginal likelihood,GP fit seconds
Function,,,,,
1,19,Matern_1.5_original,"1.07**2 * Matern(length_scale=[2, 0.0388], nu=1.5) + WhiteKernel(noise_level=5.85e-09)",-25.333946,0.388
2,19,Matern_1.5_wider,"1.07**2 * Matern(length_scale=[0.104, 10], nu=1.5) + WhiteKernel(noise_level=0.0524)",-15.993059,0.101
3,24,Matern_1.5_original,"1.26**2 * Matern(length_scale=[0.527, 2, 0.161], nu=1.5) + WhiteKernel(noise_level=0.00367)",-29.138752,0.184
4,39,Matern_1.5_wider,"4.95**2 * Matern(length_scale=[4, 3.72, 3.79, 3.89], nu=1.5) + WhiteKernel(noise_level=0.000398)",1.689692,0.228
5,29,Matern_1.5_original,"1.45**2 * Matern(length_scale=[1.83, 2, 1.42, 2], nu=1.5) + WhiteKernel(noise_level=0.011)",-8.405135,0.181
6,29,Matern_2.5_original,"1.68**2 * Matern(length_scale=[0.998, 1.34, 1.4, 1.23, 2], nu=2.5) + WhiteKernel(noise_level=0.0306)",-21.396370,0.166
7,39,Matern_2.5_original,"0.642**2 * Matern(length_scale=[0.802, 2, 2, 0.178, 0.165, 0.17], nu=2.5) + WhiteKernel(noise_level=1.65e-10)",-28.182505,0.189
8,49,Matern_2.5_wider,"2.9**2 * Matern(length_scale=[3.1, 4.78, 2.52, 4.73, 10, 5.04, 3.19, 10], nu=2.5) + WhiteKernel(noise_level=1e-10)",15.783304,0.488



All eight Gaussian Processes were fitted using observations through round 9.
Kernel configurations were retained from CP_20; continuous kernel parameters were re-estimated.

Recorded fitting warnings: 11


,Function,Category,Message
0,1,ConvergenceWarning,The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
1,2,ConvergenceWarning,The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
2,3,ConvergenceWarning,The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
3,5,ConvergenceWarning,The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
4,5,ConvergenceWarning,The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
5,6,ConvergenceWarning,The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
6,7,ConvergenceWarning,The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
7,7,ConvergenceWarning,The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
8,8,ConvergenceWarning,The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
9,8,ConvergenceWarning,The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.


### Round 10 — Gaussian Process fitting and model assumptions

This cell fits a separate Gaussian Process for each function using the
initial observations and feedback through round 9. The Matérn smoothness
settings and parameter bounds are retained from CP_20, while the amplitude,
length scales and white-noise level are re-estimated.

Outputs are normalised during fitting. Subsequent predictions are returned
in the original output units. The random seed and three optimiser restarts
are recorded explicitly to support reproducibility.

The kernel assumes that relationships between inputs can be represented
using a stationary covariance structure with a separate length scale for
each variable. This permits different sensitivities across dimensions,
but may inadequately represent abrupt changes or regions with different
local behaviour.

Retaining the configurations provides continuity across rounds, but it
does not establish that they remain optimal. In particular, the large
round 9 prediction errors for Functions 4 and 5 remain evidence of model
limitations; refitting alone does not demonstrate that these have been
resolved.

The summary records fitted kernels, log-marginal likelihoods and fitting
times. Any fitting warnings are preserved for inspection. Parameters at
their bounds can indicate constrained estimates; they do not establish
that a variable is irrelevant or that the underlying function is noiseless.

Log-marginal likelihood describes fit under the model assumptions.
It should not be treated as a substitute for predictive validation or
compared across different functions as a ranking of model quality.

### Interpreting the fitted kernels

Function 1 shows a substantial change in its fitted length scales:
from [0.02, 2] in CP_20 to approximately [2, 0.0388] in CP_21.
The new model represents faster variation along x2 and slower variation
along x1. This reversal highlights sensitivity to the updated data and
optimisation outcome; it should not be interpreted as established
feature importance.

For Functions 4 and 5, the fitted white-noise levels increased relative
to CP_20. This is consistent with the model allowing more unexplained
variation after incorporating the latest observations. It does not
identify whether that variation comes from measurement noise or an
inadequate covariance model. Noise levels are fitted on normalised
targets, so their raw values should be interpreted with that scaling
in mind.

The configurations remain unchanged, but the fitted parameters have
adapted to the new observations. Predictive reliability must still be
assessed using subsequent evaluations, particularly for Functions 4
and 5 after their large round 9 errors.

In [6]:
# CP_21 — Round 10
# Refit the auxiliary models using observations through round 9.

import time
import warnings

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

assert "updated_data" in globals(), (
    "Run the data-loading cell before fitting the auxiliary models."
)

# Retain the auxiliary model settings used in CP_20.
auxiliary_estimators = {
    5: (
        "NN (8,)",
        MLPRegressor(
            hidden_layer_sizes=(8,),
            activation="tanh",
            solver="lbfgs",
            alpha=0.1,
            max_iter=2000,
            random_state=42,
        ),
    ),
    7: (
        "Linear regression",
        LinearRegression(),
    ),
}

fitted_auxiliary_models = {}
auxiliary_fit_rows = []
auxiliary_warning_rows = []

for function, (model_name, estimator) in auxiliary_estimators.items():
    X = np.asarray(updated_data[function]["X"], dtype=float)
    y = np.asarray(updated_data[function]["y"], dtype=float)

    assert X.ndim == 2 and y.shape == (X.shape[0],), (
        f"Function {function}: incompatible input and output shapes."
    )
    assert np.all(np.isfinite(X)) and np.all(np.isfinite(y)), (
        f"Function {function}: non-finite training data."
    )

    # Standardise inputs and targets during fitting.
    # Predictions are automatically returned in original output units.
    model = TransformedTargetRegressor(
        regressor=Pipeline([
            ("input_scaler", StandardScaler()),
            ("model", estimator),
        ]),
        transformer=StandardScaler(),
    )

    start_time = time.perf_counter()

    with warnings.catch_warnings(record=True) as captured_warnings:
        warnings.simplefilter("always")
        model.fit(X, y)

    fit_seconds = time.perf_counter() - start_time

    fitted_auxiliary_models[function] = model

    auxiliary_fit_rows.append({
        "Function": function,
        "Observations": len(y),
        "Auxiliary model": model_name,
        "Fit seconds": fit_seconds,
    })

    for warning in captured_warnings:
        auxiliary_warning_rows.append({
            "Function": function,
            "Category": warning.category.__name__,
            "Message": str(warning.message),
        })

auxiliary_models = fitted_auxiliary_models

auxiliary_fit_summary = (
    pd.DataFrame(auxiliary_fit_rows)
    .set_index("Function")
)

auxiliary_fit_warnings = pd.DataFrame(
    auxiliary_warning_rows,
    columns=["Function", "Category", "Message"],
)

display(
    auxiliary_fit_summary.style.format({
        "Fit seconds": "{:.3f}",
    })
)

print(
    "\nAuxiliary models fitted for Functions 5 and 7 "
    "using observations through round 9."
)
print("Predictions will be returned in the original output units.")

if auxiliary_fit_warnings.empty:
    print("No auxiliary fitting warnings were recorded.")
else:
    print(
        f"\nRecorded auxiliary fitting warnings: "
        f"{len(auxiliary_fit_warnings)}"
    )
    with pd.option_context("display.max_colwidth", None):
        display(auxiliary_fit_warnings)

,Observations,Auxiliary model,Fit seconds
Function,,,
5,29,"NN (8,)",0.046
7,39,Linear regression,0.006



Auxiliary models fitted for Functions 5 and 7 using observations through round 9.
Predictions will be returned in the original output units.
No auxiliary fitting warnings were recorded.


### Round 10 — Auxiliary models

This cell refits the auxiliary neural network for Function 5 using
29 observations and the linear regression model for Function 7 using
39 observations. Their settings are retained from CP_20.

Inputs and targets are standardised during fitting. Predictions are
automatically transformed back into the original output units, making
them directly comparable with GP predictions and observed outputs.

In round 9, the neural network predicted Function 5 more accurately
than the GP, whereas linear regression was less accurate than the GP
for Function 7. These are individual observations and do not establish
general superiority of either model.

The auxiliary predictions provide additional evidence about model
agreement. They do not directly determine query selection: the selection
rule remains GP-UCB. Agreement between models does not guarantee
accuracy, since both models are trained on the same limited and
adaptively sampled data.

In [7]:
# CP_21 — Round 10
# Compare global and local candidates using GP-UCB.

import time

import numpy as np
import pandas as pd

from IPython.display import display
from scipy.spatial import cKDTree

assert "gp_models" in globals(), "Run the GP fitting cell first."
assert "auxiliary_models" in globals(), (
    "Run the auxiliary model fitting cell first."
)
assert completed_rounds == 9 and submission_round == 10, (
    "This cell prepares round 10 using nine completed feedback rounds."
)

ucb_weight = 2.0
global_candidate_count = 100_000
local_candidate_count = 10_000
local_radii = [0.10, 0.03]
prediction_batch_size = 5_000

lower_bound = 0.0
upper_bound = 0.999999
duplicate_tolerance = 1e-6

round_10_candidate_proposals = {}
candidate_rows = []
search_time_rows = []


def predict_in_batches(model, candidates, batch_size):
    """Calculate GP predictions without processing all candidates at once."""
    means = []
    standard_deviations = []

    for start in range(0, len(candidates), batch_size):
        batch = candidates[start:start + batch_size]
        mean, std = model.predict(batch, return_std=True)
        means.append(mean)
        standard_deviations.append(std)

    return np.concatenate(means), np.concatenate(standard_deviations)


for function in range(1, 9):
    start_time = time.perf_counter()

    X = np.asarray(updated_data[function]["X"], dtype=float)
    y = np.asarray(updated_data[function]["y"], dtype=float)
    model = gp_models[function]

    dimension = X.shape[1]
    best_index = int(np.argmax(y))
    best_input = X[best_index]
    current_best = float(y[best_index])

    rng = np.random.default_rng(42 + function)
    observation_tree = cKDTree(X)

    candidate_sets = {
        "Global": rng.uniform(
            lower_bound,
            upper_bound,
            size=(global_candidate_count, dimension),
        )
    }

    # Local regions are boxes centred on the best observed input.
    # Bounds are restricted before sampling, avoiding clipping duplicates.
    for radius in local_radii:
        local_lower = np.maximum(lower_bound, best_input - radius)
        local_upper = np.minimum(upper_bound, best_input + radius)

        candidate_sets[f"Local {radius:.2f}"] = rng.uniform(
            local_lower,
            local_upper,
            size=(local_candidate_count, dimension),
        )

    round_10_candidate_proposals[function] = {}

    for search_name, candidates in candidate_sets.items():
        # Evaluate the same six-decimal coordinates used for submission.
        candidates = np.unique(
            np.round(candidates, 6),
            axis=0,
        )

        nearest_distances, _ = observation_tree.query(candidates, k=1)

        keep = nearest_distances > duplicate_tolerance
        candidates = candidates[keep]
        nearest_distances = nearest_distances[keep]

        assert len(candidates) > 0, (
            f"Function {function}, {search_name}: no valid candidates."
        )
        assert np.all(
            (candidates >= lower_bound) & (candidates <= upper_bound)
        ), "Candidate coordinates are outside the permitted bounds."

        means, standard_deviations = predict_in_batches(
            model,
            candidates,
            prediction_batch_size,
        )

        assert np.all(np.isfinite(means)), "Non-finite GP means."
        assert np.all(np.isfinite(standard_deviations)), (
            "Non-finite GP standard deviations."
        )

        ucb_scores = means + ucb_weight * standard_deviations
        selected_index = int(np.argmax(ucb_scores))
        query = candidates[selected_index].copy()

        auxiliary_prediction = np.nan
        if function in auxiliary_models:
            auxiliary_prediction = float(
                auxiliary_models[function].predict(
                    query.reshape(1, -1)
                )[0]
            )

        proposal = {
            "query": query,
            "mean": float(means[selected_index]),
            "std": float(standard_deviations[selected_index]),
            "ucb": float(ucb_scores[selected_index]),
            "nearest_distance": float(
                nearest_distances[selected_index]
            ),
            "auxiliary_prediction": auxiliary_prediction,
        }

        round_10_candidate_proposals[function][search_name] = proposal

        candidate_rows.append({
            "Function": function,
            "Search": search_name,
            "Query": "-".join(f"{value:.6f}" for value in query),
            "Current best": current_best,
            "GP mean": proposal["mean"],
            "GP std": proposal["std"],
            "UCB": proposal["ucb"],
            "Nearest observation distance": proposal["nearest_distance"],
            "Auxiliary prediction": auxiliary_prediction,
        })

    search_time_rows.append({
        "Function": function,
        "Candidate search seconds": time.perf_counter() - start_time,
    })


round_10_candidates = pd.DataFrame(candidate_rows)

candidate_search_times = (
    pd.DataFrame(search_time_rows)
    .set_index("Function")
)

display(
    round_10_candidates.style.format(
        {
            "Current best": "{:.6e}",
            "GP mean": "{:.6e}",
            "GP std": "{:.6e}",
            "UCB": "{:.6e}",
            "Nearest observation distance": "{:.6f}",
            "Auxiliary prediction": "{:.6e}",
        },
        na_rep="—",
    )
)

display(
    candidate_search_times.style.format({
        "Candidate search seconds": "{:.3f}",
    })
)

print("\nGenerated three proposals per function: 24 proposals in total.")
print("Each proposal maximises UCB within its sampled candidate set.")
print("Final round 10 queries have not yet been selected.")

,Function,Search,Query,Current best,GP mean,GP std,UCB,Nearest observation distance,Auxiliary prediction
0,1,Global,0.979455-0.999881,7.710875e-16,-1.009432e-04,7.547547e-04,1.408566e-03,0.326971,—
1,1,Local 0.10,0.830609-0.812268,7.710875e-16,-1.097589e-04,7.445014e-04,1.379244e-03,0.127282,—
2,1,Local 0.03,0.761008-0.749102,7.710875e-16,1.699119e-04,3.028127e-04,7.755374e-04,0.034034,—
3,2,Global,0.711503-0.000155,6.582395e-01,5.881763e-01,6.743702e-02,7.230503e-01,0.009855,—
4,2,Local 0.10,0.713175-0.000276,6.582395e-01,5.871169e-01,6.800529e-02,7.231274e-01,0.008267,—
5,2,Local 0.03,0.713020-0.000033,6.582395e-01,5.872322e-01,6.795209e-02,7.231363e-01,0.008511,—
6,3,Global,0.317762-0.998895-0.417393,-3.096391e-02,-3.501582e-02,4.632375e-02,5.763168e-02,0.161226,—
7,3,Local 0.10,0.999880-0.871291-0.445814,-3.096391e-02,-3.896675e-02,2.689122e-02,1.481570e-02,0.110835,—
8,3,Local 0.03,0.996817-0.980558-0.475655,-3.096391e-02,-3.131183e-02,1.817607e-02,5.040304e-03,0.033906,—
9,4,Global,0.378762-0.416286-0.401546-0.396799,6.152729e-01,1.038839e-01,2.663174e-01,6.365188e-01,0.048808,—


,Candidate search seconds
Function,
1,0.410
2,0.319
3,0.366
4,0.432
5,0.393
6,0.492
7,0.705
8,0.701



Generated three proposals per function: 24 proposals in total.
Each proposal maximises UCB within its sampled candidate set.
Final round 10 queries have not yet been selected.


### Round 10 — Global and local candidate search

For each function, the search generates 100,000 global candidates and
10,000 candidates within each of two local boxes centred on the best
observed input. The local boxes extend by up to 0.10 and 0.03 in each
coordinate, restricted to the permitted domain.

Candidates are rounded to six decimal places before prediction. Duplicate
candidate rows are removed, and points within a Euclidean distance of
1e-6 from an existing observation are excluded. This exclusion prevents
duplicate or extremely close evaluations; it does not guarantee that
the remaining candidates provide substantial new information.

Each candidate receives a UCB score equal to the GP predicted mean plus
twice its predictive standard deviation. The highest-scoring candidate
from each search set is retained, producing three proposals per function.
These are maxima within the sampled sets, not proven maxima over the
continuous search space.

The table separates predicted performance, uncertainty, UCB and distance
to existing observations. Auxiliary predictions for Functions 5 and 7
are reported for comparison and do not alter the UCB ranking.

The fixed seeds and explicit search settings support reproducibility.
However, reusing the seeds also reuses the global candidate locations
across rounds. Local sampling concentrates on previously successful
regions, and even 100,000 global candidates provide limited coverage
in higher dimensions. These choices are documented as potential sources
of search bias.

The labels "Global" and "Local" describe how candidates were generated.
Both searches balance predicted performance and uncertainty through UCB;
a global proposal is not necessarily purely exploratory, and a local
proposal is not necessarily purely exploitative.

### Round 10 — Interpreting the candidate comparison

The highest-UCB proposals come from the global search for Functions 1
and 3, the local 0.03 search for Functions 2, 4, 5 and 6, and the local
0.10 search for Functions 7 and 8.

- **Function 1:** The global proposal has a negative predicted mean but
  substantial predictive uncertainty. Its selection is driven by the
  uncertainty contribution, following a substantial change in the fitted
  length scales. It is an exploratory choice under an unstable model.

- **Function 2:** All three proposals are concentrated near x1 = 0.71
  and x2 = 0. Their UCB scores are very similar. The local 0.03 proposal
  wins numerically, but the small difference does not establish a
  meaningful advantage in actual performance.

- **Function 3:** The global proposal has greater uncertainty and the
  highest UCB. Its predicted mean remains below the current best, so the
  choice tests potential improvement rather than relying on a predicted
  record output.

- **Function 4:** The local 0.03 proposal has the highest UCB and lies
  approximately 0.015 from an existing observation. This continues local
  refinement, while the large round 9 prediction error limits confidence
  in the predicted benefit.

- **Function 5:** The local 0.03 proposal is only approximately 0.000235
  from an existing observation. It passes the numerical duplicate filter
  but is effectively a very close revisit. Both the GP and neural network
  predict an output below the current best. The uncertainty contribution
  raises UCB above that best value; agreement between the models does
  not guarantee improvement.

- **Function 6:** The local 0.03 proposal has the highest, least negative
  UCB. Its predicted mean remains below the current best. Negative
  outputs are compared using the same maximisation rule as other functions.

- **Function 7:** The local 0.10 proposal narrowly exceeds the global
  proposal in UCB. The auxiliary linear prediction is substantially
  lower than the GP prediction, highlighting model disagreement.
  The auxiliary prediction does not change the selection rule.

- **Function 8:** After improving in round 9, the local search is centred
  on the new best observed input. The local 0.10 proposal wins through
  a balance of predicted performance and uncertainty, although its mean
  remains below the current best.

None of the selected GP means exceeds its function's current best.
Selection therefore depends on the uncertainty contribution in every
case. UCB is an acquisition score, not a guaranteed upper bound on the
next observed output.

The Function 5 near-revisit also exposes a limitation of the current
strategy: preventing exact duplicates does not ensure informative spatial
exploration. With a WhiteKernel, predictive standard deviation includes
a fitted observation-noise contribution, so it should not be interpreted
entirely as reducible uncertainty about the function.

In [8]:
# CP_21 — Round 10
# Select the highest-UCB proposal for each function.
# Validate and display the final queries for manual portal submission.

import re

import numpy as np
import pandas as pd

from IPython.display import display
from scipy.spatial import cKDTree

required_variables = [
    "round_10_candidate_proposals",
    "updated_data",
    "expected_dimensions",
    "duplicate_tolerance",
    "ucb_weight",
    "submission_round",
]

missing_variables = [
    name for name in required_variables if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the preceding cells first. "
        f"Missing variables: {', '.join(missing_variables)}"
    )

assert submission_round == 10, "This cell prepares round 10."

assert set(round_10_candidate_proposals) == set(range(1, 9)), (
    "Expected candidate proposals for all eight functions."
)

round_10_selected_proposals = {}
selected_rows = []
submission_rows = []

for function in range(1, 9):
    proposals = round_10_candidate_proposals[function]

    assert len(proposals) == 3, (
        f"Function {function}: expected three search proposals."
    )

    assert all(
        np.isfinite(proposal["ucb"])
        for proposal in proposals.values()
    ), f"Function {function}: non-finite UCB score."

    # Compare full numerical scores, not rounded display values.
    selected_search = max(
        proposals,
        key=lambda search: proposals[search]["ucb"],
    )

    proposal = proposals[selected_search]
    query = np.asarray(proposal["query"], dtype=float).copy()

    dimension = expected_dimensions[function]

    assert query.shape == (dimension,), (
        f"Function {function}: incorrect query dimensionality."
    )
    assert np.all(np.isfinite(query)), (
        f"Function {function}: non-finite coordinates."
    )
    assert np.all((query >= 0.0) & (query <= 0.999999)), (
        f"Function {function}: coordinates outside submission bounds."
    )

    query_string = "-".join(f"{value:.6f}" for value in query)

    pattern = rf"0\.\d{{6}}(?:-0\.\d{{6}}){{{dimension - 1}}}"
    assert re.fullmatch(pattern, query_string), (
        f"Function {function}: invalid submission format."
    )

    submitted_query = np.asarray(
        [float(value) for value in query_string.split("-")],
        dtype=float,
    )

    assert np.allclose(
        submitted_query,
        query,
        rtol=0,
        atol=1e-12,
    ), (
        f"Function {function}: formatting changed the evaluated query."
    )

    X = np.asarray(updated_data[function]["X"], dtype=float)
    nearest_distance = float(
        cKDTree(X).query(submitted_query, k=1)[0]
    )

    assert nearest_distance > duplicate_tolerance, (
        f"Function {function}: query duplicates or is too close "
        "to an existing observation under the configured tolerance."
    )

    assert np.isclose(
        proposal["ucb"],
        proposal["mean"] + ucb_weight * proposal["std"],
        rtol=1e-10,
        atol=1e-12,
    ), f"Function {function}: inconsistent UCB calculation."

    round_10_selected_proposals[function] = {
        **proposal,
        "query": submitted_query.copy(),
        "search": selected_search,
        "query_string": query_string,
        "nearest_distance": nearest_distance,
    }

    selected_rows.append({
        "Function": function,
        "Selected search": selected_search,
        "Query": query_string,
        "Predicted mean": proposal["mean"],
        "Predicted std": proposal["std"],
        "UCB": proposal["ucb"],
        "Nearest observation distance": nearest_distance,
        "Auxiliary prediction": proposal["auxiliary_prediction"],
    })

    submission_rows.append({
        "Function": function,
        "Query for submission": query_string,
    })

round_10_selected = (
    pd.DataFrame(selected_rows)
    .set_index("Function")
)

round_10_submission = (
    pd.DataFrame(submission_rows)
    .set_index("Function")
)

display(
    round_10_selected.style.format(
        {
            "Predicted mean": "{:.6e}",
            "Predicted std": "{:.6e}",
            "UCB": "{:.6e}",
            "Nearest observation distance": "{:.6f}",
            "Auxiliary prediction": "{:.6e}",
        },
        na_rep="—",
    )
)

display(round_10_submission)

print()

for function, row in round_10_submission.iterrows():
    print(f"Function {function}: {row['Query for submission']}")

print("\nAll eight queries passed validation.")
print(
    "Validation confirms format, bounds and the configured "
    "duplicate tolerance; it does not guarantee improvement."
)
print("Queries are prepared; this code does not submit them to the portal.")

,Selected search,Query,Predicted mean,Predicted std,UCB,Nearest observation distance,Auxiliary prediction
Function,,,,,,,
1,Global,0.979455-0.999881,-1.009432e-04,7.547547e-04,1.408566e-03,0.326971,—
2,Local 0.03,0.713020-0.000033,5.872322e-01,6.795209e-02,7.231363e-01,0.008511,—
3,Global,0.317762-0.998895-0.417393,-3.501582e-02,4.632375e-02,5.763168e-02,0.161226,—
4,Local 0.03,0.409023-0.410500-0.383078-0.419599,3.606262e-01,2.124775e-01,7.855813e-01,0.014990,—
5,Local 0.03,0.997181-0.999005-0.999356-0.996804,8.045722e+03,3.267279e+02,8.699178e+03,0.000235,8.054748e+03
6,Local 0.03,0.459114-0.364921-0.534349-0.679924-0.000139,-3.720083e-01,1.126321e-01,-1.467440e-01,0.017641,—
7,Local 0.10,0.021409-0.225288-0.479072-0.145922-0.286321-0.697760,2.280249e+00,1.273900e-01,2.535030e+00,0.113450,1.092263e+00
8,Local 0.10,0.071062-0.278596-0.059377-0.031881-0.988708-0.628793-0.187893-0.999301,9.944405e+00,2.760331e-02,9.999611e+00,0.185514,—


,Query for submission
Function,
1,0.979455-0.999881
2,0.713020-0.000033
3,0.317762-0.998895-0.417393
4,0.409023-0.410500-0.383078-0.419599
5,0.997181-0.999005-0.999356-0.996804
6,0.459114-0.364921-0.534349-0.679924-0.000139
7,0.021409-0.225288-0.479072-0.145922-0.286321-0...
8,0.071062-0.278596-0.059377-0.031881-0.988708-0...



Function 1: 0.979455-0.999881
Function 2: 0.713020-0.000033
Function 3: 0.317762-0.998895-0.417393
Function 4: 0.409023-0.410500-0.383078-0.419599
Function 5: 0.997181-0.999005-0.999356-0.996804
Function 6: 0.459114-0.364921-0.534349-0.679924-0.000139
Function 7: 0.021409-0.225288-0.479072-0.145922-0.286321-0.697760
Function 8: 0.071062-0.278596-0.059377-0.031881-0.988708-0.628793-0.187893-0.999301

All eight queries passed validation.
Validation confirms format, bounds and the configured duplicate tolerance; it does not guarantee improvement.
Queries are prepared; this code does not submit them to the portal.


### Round 10 — Final query selection and validation

One query per function is selected using the highest full-precision UCB
among the three search proposals. The selected coordinates and predictions
remain available in notebook variables for comparison with future feedback.

Validation checks dimensionality, finite values, permitted coordinate
bounds, six-decimal formatting and separation from existing observations
under the configured tolerance. Passing these checks establishes technical
validity, not predictive reliability or expected information gain.

The Function 5 query remains a very close revisit. It is retained because
it has the highest UCB under the stated rule, with this limitation explicitly
documented rather than hidden by the successful validation message.

The submission table contains only the input coordinates to enter into
the portal. Predicted outputs and UCB scores are not observed results.

### Dataset datasheet — Observations available before round 10

**Purpose and provenance**

These datasets support the sequential maximisation of eight unknown
functions in the capstone challenge. They combine the initial observations
provided by the programme with input–output pairs returned by the portal
for nine completed rounds.

The original sampling procedure and the underlying meanings of the
variables are not documented in the files used here. Function definitions,
global optima and the evaluation noise mechanism are unknown.

**Composition**

| Function | Input variables | Initial observations | Feedback observations | Total |
|---|---:|---:|---:|---:|
| 1 | 2 | 10 | 9 | 19 |
| 2 | 2 | 10 | 9 | 19 |
| 3 | 3 | 15 | 9 | 24 |
| 4 | 4 | 30 | 9 | 39 |
| 5 | 4 | 20 | 9 | 29 |
| 6 | 5 | 20 | 9 | 29 |
| 7 | 6 | 30 | 9 | 39 |
| 8 | 8 | 40 | 9 | 49 |

Each observation consists of an input vector and one scalar output.
Larger outputs are preferred, including when all observed outputs are
negative. Output scales differ substantially between functions.

**Loading and preprocessing**

The notebook checks feedback round counts, input dimensions, finite values,
scalar outputs and permitted feedback coordinate ranges. Initial datasets
are checked for expected shapes and finite values.

Datasets are rebuilt from the initial files and cumulative feedback on
each execution, preventing repeated appending of the same rounds.
Negative and unsuccessful observations are retained.

Raw outputs remain available in their original units. Normalisation or
standardisation is performed within the fitted models.

**Sampling and coverage limitations**

Feedback observations were selected adaptively using previous results.
They are not independent, uniformly sampled test observations. Local
search concentrates evaluations around previously successful inputs,
while other regions remain sparsely explored.

Coverage is particularly limited in higher-dimensional functions.
Sampling bias here concerns coverage of the numerical search space;
the available data do not support conclusions about demographic fairness.

**Maintenance**

Round 10 queries are proposals, not observations. Their outputs will only
be added after portal feedback is received and matched to the submitted
coordinates. Initial data, completed feedback and pending queries should
remain distinguishable.

### Model card — Round 10 optimisation workflow

**Intended use**

The models guide the selection of one new input per function within the
capstone domain. They are surrogate models of unknown numerical functions.
Their predictions are not guarantees of improvement or explanations of
the underlying functions' causal mechanisms.

**Models and training**

Each function uses a separate Gaussian Process with a constant-amplitude
Matérn kernel, one length scale per input variable and a WhiteKernel.

Matérn smoothness and parameter bounds are retained from CP_20.
Continuous kernel parameters are re-estimated using all observations
through round 9. Outputs are normalised during GP fitting, with predictions
returned in original units. The optimiser uses three restarts and a
random seed of 42.

Function 5 also uses a neural network with one hidden layer of eight
units. Function 7 uses auxiliary linear regression. Both standardise
inputs and targets internally. Their predictions are reported for
comparison and do not alter GP-UCB selection.

**Query selection**

For each function, the notebook samples 100,000 global candidates and
10,000 candidates from each of two local boxes around the best observed
input. Local half-widths are 0.10 and 0.03 per coordinate.

The candidate generator uses seed 42 plus the function number.
Coordinates are rounded to six decimal places before evaluation.
Candidates within Euclidean distance 1e-6 of an existing observation
are excluded.

The acquisition score is:

UCB = GP predicted mean + 2 × GP predictive standard deviation.

The highest-UCB candidate across the three search proposals is selected.
This is an optimum within the sampled candidate sets, not a proven global
optimum over the continuous domain.

**Evaluation evidence**

Round 9 predictions were checked against the corresponding portal outputs
before refitting. Functions 4 and 5 had absolute errors of approximately
3.20 and 6.36 predicted standard deviations, respectively.

Only Function 8 improved its previous best output in round 9.
These sequential observations provide useful diagnostics but are not
a representative independent test set. No new cross-validation comparison
of model configurations was performed in CP_21.

**Known limitations**

The GP covariance assumptions may inadequately describe abrupt changes
or varying local behaviour. Eleven fitting warnings identify parameters
near their bounds. Function 1 also shows a reversal in its fitted
length-scale pattern compared with CP_20.

The selected Function 5 query lies approximately 0.000235 from an existing
observation. It passes the duplicate filter but offers very little spatial
novelty. Predictive uncertainty includes a fitted white-noise contribution
and should not be interpreted entirely as reducible uncertainty.

All eight selected GP means are below their current best observations.
Their selection depends on the uncertainty contribution to UCB.

**Reproducibility requirements**

Reproduction requires the initial files, cumulative feedback through
round 9, this notebook, the stated seeds and settings, and compatible
Python and package versions. Paths must be adapted to the researcher's
environment.

The notebook retains candidate scores, selected queries, predictions and
warnings. Saving the executed notebook preserves the displayed results;
in-memory model objects do not survive a kernel restart.